# 032 — CGC ESI Controller Test (firmware/DLL 1-00)

Real-hardware gate for the P9 class rework. Exercises the reworked `ESI`
class against firmware 1-00: **configuration selection, HV voltages
(modules 2 + 3), heater temperature, measurement ranges** — plus the open
questions from the rework.

**DLL bridge (2026-07-23):** the `x64/` DLL is **broken** (manufacturer
statement). The working DLL is the 32-bit build directly in
`ESI-CTRL_1-00/`; `ESIBase` now reaches it through a 32-bit bridge
process (msl-loadlib, `pip install msl-loadlib` — already in
pyproject). First `ESI(...)` construction starts the bridge server:
expect a few extra seconds once per kernel.

**Operator workflow (CGC email 2026-07-21, user-verified 2026-07-23):**
load an NVM configuration — it takes effect **immediately** (heating
starts, HV is applied; no further enable step) — then tweak the heater
temperature and the HV target voltages live.

**Slot numbering (hardware observation 2026-07-23):** the DLL slots are
**0-based**: cfg-file section `[ConfigurationN]` = DLL slot **N−1**
(device list shows 0='Off', 1='Standby', 9='Heat 30deg', 11='Heat 50deg',
100='HV1 +100V'). All slot numbers below use **DLL numbering**.

Device NVM (user cfg, loaded via vendor tool):

| DLL slot | Name | Notes |
|---|---|---|
| 0 | Off | everything disabled |
| 1 | Standby | DeviceEnable=N, HV modules enabled, heater limit **180 W** |
| 9–24 | Heat 30…175deg | DeviceEnable=Y; working slots may carry preset HV voltages |
| 99–179 | HV1 presets | +0…+3000 V / −0…−3000 V |
| 199–279 | HV2 presets | analogous |

**Config-file gotchas (user 2026-07-23):**
- `InterlockEnable` must be exactly **`Y,N,N,Y`** — the 2026-07-23 run
  sat in `STATE_ERR_ILOCK` the whole time with mask readback 7; an
  interlock error blocks `ST_ON`, so fix this **first**.
- `HVPSxMaxVoltStep` must be **nonzero** (good value 10 V) — at 0 **no
  voltage gets applied at all**.
- HVPS1 slot has no module; lab modules are HVPS2/HVPS3 on addresses
  2/3 (mapping probe below confirms).

**Open questions this notebook answers on hardware:**
1. What flips `ST_ON` ↔ `ST_STBY`? (assumed: `DeviceEnable` of the loaded
   config; also test `set_enable()` alone) — 2026-07-23 run inconclusive,
   device was stuck in `STATE_ERR_ILOCK`.
2. Preset mapping: cfg keys `HVPS1..3` vs lab module addresses **2**
   (inlet) and **3** (emitter) — which address does "HV1 +100V" hit?
3. Numeric interlock mask ↔ `Y,N,N,Y` mapping (read back after a working
   config is loaded).
4. Do the two 1-00 signature fixes hold on hardware
   (`get_base_housekeeping` leading `Valid`, `get_complete_state` 9 args)?

**Safety**
- ⚠️ SINGLE-INSTANCE DLL: never run the Explorer ESI plugin and this
  notebook at the same time (second connect fails loudly by design).
- ⚠️ Heater: keep the power limit at **10–30 W below 50 °C** (overshoot).
  Slot 1 "Standby" leaves 180 W configured — the manual-heater cell sets
  20 W first.
- Rule: `# CHANGE ESI:` comment before every hardware-parameter block.


## Setup

In [1]:
import os
import time
import logging
from pathlib import Path
from datetime import datetime

from devices.cgc.esi.esi import ESI

repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"032_cgc_esi_100_test_{timestamp}.log"

logger = logging.getLogger(f"032_cgc_esi_100_test_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
console_handler.setLevel(logging.DEBUG)
logger.addHandler(console_handler)

print(f"Logging to {log_file}")

Logging to C:\Users\ESIBDlab\Desktop\LAB_Code\esibd_bs\debugging\logs\032_cgc_esi_100_test_20260727_151641.log


In [2]:
COM_PORT = 14  # lab_config.toml [com_ports] ESI = 14

# First construction starts the 32-bit DLL bridge server (msl-loadlib) —
# a few seconds once per kernel.
esi = ESI(device_id="esi_100_test", com=COM_PORT, logger=logger)

INFO - esi_100_test  COM14  initialized (230400 baud)


In [3]:
# CHANGE ESI: connect — bring-up runs open_port -> set_comspeed(230400) -> set_enable(True)
esi.connect()

INFO - esi_100_test  COM14  comspeed set to 230400
INFO - esi_100_test  COM14  setting device enable to True
INFO - esi_100_test  COM14  connected (vendor DLL, 230400 baud)


True

## Identity & modules

Firmware/DLL versions and the module map. Expected: address 0 = heat
controller HTCTRL-24-10 (`0xDB1C`, **new** — now installed), addresses
2 + 3 = HVPS-3kB (`0x0A0D`), address 1 empty.

In [4]:
print("DLL sw version :", esi.get_sw_version())
print("fw version     :", esi.get_fw_version())
print("fw date        :", esi.get_fw_date())
print("product id     :", esi.get_product_id())
print("product no     :", esi.get_product_no())
print("hw type/version:", esi.get_hw_type(), esi.get_hw_version())
print("uptime         :", esi.get_uptime())

DLL sw version : 256
fw version     : (0, 256)
fw date        : (0, 'Jul 13 2026')
product id     : (0, 'ESI Controller')
product no     : (0, 124701)
hw type/version: (0, 4333632) (0, 256)
uptime         : (0, 5584.072265625, 4340938.44921875)


In [5]:
status, valid, max_module, presence = esi.get_module_presence()
print(f"presence status={status} valid={valid} max_module={max_module}")
labels = {esi.MODULE_NOT_FOUND: "not found", esi.MODULE_PRESENT: "PRESENT",
          esi.MODULE_INVALID: "invalid"}
for addr in range(esi.MODULE_NUM):
    line = f"  addr {addr}: {labels.get(presence[addr], presence[addr])}"
    if presence[addr] == esi.MODULE_PRESENT:
        st, dev_type = esi.get_module_dev_type(addr)
        kind = {esi.MODULE_HTCTRL_TYPE: "HTCTRL", esi.MODULE_HVPS_TYPE: "HVPS",
                esi.MODULE_BASE_TYPE: "BASE"}.get(dev_type, hex(dev_type))
        line += f"  type={kind} ({hex(dev_type)})  fw={esi.get_module_fw_version(addr)}"
    print(line)
print(f"  base module: {labels.get(presence[esi.PRESENCE_BASE], '?')}")

presence status=0 valid=True max_module=4
  addr 0: PRESENT  type=HTCTRL (0xdb1c)  fw=(0, 256)
  addr 1: not found
  addr 2: PRESENT  type=HVPS (0xa0d)  fw=(0, 256)
  addr 3: PRESENT  type=HVPS (0xa0d)  fw=(0, 256)
  base module: PRESENT


## Baseline state reads

Includes the two 1-00 signature changes: `get_base_housekeeping` now has a
leading `Valid`, `get_complete_state` lost the trailing heat-interlock arg
(9 args). Garbage values / status ≠ 0 here would mean the ctypes
signatures are wrong → **stop**.

In [6]:
print("main state     :", esi.get_main_state())
print("enable         :", esi.get_enable())
print("device state   :", esi.get_device_state())
print("voltage state  :", esi.get_voltage_state())
print("temp state     :", esi.get_temperature_state())
print("interlock state:", esi.get_interlock_state())
# 2026-07-23 run: mask readback 7 -> STATE_ERR_ILOCK (blocks ST_ON).
# Required lab value = cfg 'Y,N,N,Y'; numeric mapping recorded below
# after a working config is loaded.
print("interlock enab :", esi.get_interlock_enable())
print("fan state      :", esi.get_fan_state())

main state     : (0, '0x0', 'STATE_ON')
enable         : (0, True)
device state   : (0, '0x0', ['DEVST_OK'])
voltage state  : (0, '0x37', ['VS_3V3_OK', 'VS_5V0_OK', 'VS_24V_OK', 'VS_LINE_OK', 'VS_PSU_OK'])
temp state     : (0, '0x0', [])
interlock state: (0, '0xfa0e', ['IS_HTCTRL_ILOCK2', 'IS_CTRL_ILOCK_FP', 'IS_CTRL_ILOCK_RP', 'IS_HTCTRL_ILOCK2_CURR', 'IS_HTCTRL_ILOCK2_LAST', 'IS_CTRL_ILOCK_FP_CURR', 'IS_CTRL_ILOCK_RP_CURR', 'IS_CTRL_ILOCK_FP_LAST', 'IS_CTRL_ILOCK_RP_LAST'])
interlock enab : (0, 0)
fan state      : (0, '0xf', ['FS_FAN_OK', 'FS_FAN_SW_CURR', 'FS_FAN_SW_LAST', 'FS_FAN_ENB'])


In [7]:
# NEW 1-00 signature: (status, valid, volt_3v3, temp_cpu)
print("base hk        :", esi.get_base_housekeeping())
# (status, v24, v5, v3, temp_cpu, temp_psu)
print("housekeeping   :", esi.get_housekeeping())
print("heat-ctrl hk   :", esi.get_heat_ctrl_housekeeping())
for addr in (2, 3):
    print(f"HV{addr} hk        :", esi.get_hv_supply_housekeeping(addr))

base hk        : (0, True, 3.2981530343007917, 27.01)
housekeeping   : (0, 24.109, 5.009, 3.3040000000000003, 26.22, 28.02)
heat-ctrl hk   : (0, True, 3.3003300330033003, 27.21, 4.99, 24.042, 26.39)
HV2 hk        : (0, True, 3.2948929159802307, 28.12, 4.985, 24.034, -21.541610503719234, 17.942, -17.756477459937095, 1.504, 0.024961309969547204)
HV3 hk        : (0, True, 3.2948929159802307, 27.5, 4.987, 24.032, -21.60451300484249, 17.945, -17.87429484299336, 1.496, 0.018970595576855872)


In [8]:
# NEW 1-00 shape: 9 args, no trailing HeatCtrlInterlockState
(status, data_flags, dev_state, volt_state, temp_state, fan_state,
 ilock_state, state, mod_data_flags, mod_state) = esi.get_complete_state()
print(f"status={status} state={hex(state)} dev={hex(dev_state)} "
      f"volt={hex(volt_state)} temp={hex(temp_state)} fan={hex(fan_state)} "
      f"ilock={hex(ilock_state)}")
for addr, ms in enumerate(mod_state):
    active = []
    if ms & esi.MS_CTRL_ACT:
        active.append("CTRL_ACT")
    if ms & esi.MS_MOD_ACT:
        active.append("MOD_ACT")
    if ms & esi.MS_DEV_ACT:
        active.append("DEV_ACT")
    print(f"  module[{addr}] state={hex(ms)} {' '.join(active)}")

status=0 state=0x0 dev=0x0 volt=0x37 temp=0x0 fan=0xf ilock=0xfa0e
  module[0] state=0x8100 CTRL_ACT DEV_ACT
  module[1] state=0x0 
  module[2] state=0x8000 DEV_ACT
  module[3] state=0x8000 DEV_ACT
  module[4] state=0x8000 DEV_ACT


## Configuration management (new in 1-00)

Expected: `get_config_values` → (0, 1023, 53, 202); 42 active slots.
Observed 2026-07-23: **DLL slots are 0-based** (0='Off', 1='Standby',
9–24 heat, 99–179 HV1, 199–279 HV2) — cfg-file `[ConfigurationN]` =
DLL slot N−1. The device NVM holds the **user's own cfg** (loaded via
vendor tool), so trust this listing over any file.

In [9]:
print("config values  :", esi.get_config_values())  # (status, max_no, data_size, name_size)

status, active_slots, valid_slots = esi.list_configs()
print(f"status={status}  {len(active_slots)} active, {len(valid_slots)} valid")
for n in active_slots:
    _, name = esi.get_config_name(n)
    print(f"  slot {n:4d}  {name!r}")

config values  : (0, 1023, 53, 202)
status=0  261 active, 261 valid
  slot    0  'Off'
  slot    1  'Standby'
  slot    9  'Heat 30deg'
  slot   10  'Heat 40deg'
  slot   11  'Heat 50deg'
  slot   12  'Heat 60deg'
  slot   13  'Heat 70deg'
  slot   14  'Heat 80deg'
  slot   15  'Heat 90deg'
  slot   16  'Heat 100deg'
  slot   17  'Heat 110deg'
  slot   18  'Heat 120deg'
  slot   19  'Heat 130deg'
  slot   20  'Heat 140deg'
  slot   21  'Heat 150deg'
  slot   22  'Heat 160deg'
  slot   23  'Heat 170deg'
  slot   24  'Heat 175deg'
  slot   25  'ILOCK HV=NNNN HT=YNNN'
  slot   26  'ILOCK HV=YNNN HT=YNNN'
  slot   27  'ILOCK HV=NYNN HT=YNNN'
  slot   28  'ILOCK HV=YYNN HT=YNNN'
  slot   29  'ILOCK HV=NNYN HT=YNNN'
  slot   30  'ILOCK HV=YNYN HT=YNNN'
  slot   31  'ILOCK HV=NYYN HT=YNNN'
  slot   32  'ILOCK HV=YYYN HT=YNNN'
  slot   33  'ILOCK HV=NNNY HT=YNNN'
  slot   34  'ILOCK HV=YNNY HT=YNNN'
  slot   35  'ILOCK HV=NYNY HT=YNNN'
  slot   36  'ILOCK HV=YYNY HT=YNNN'
  slot   37  'ILOCK H

In [10]:
# Current working config as raw 53-byte blob (read-only peek)
status, blob = esi.get_current_config()
print(f"status={status}  len={len(blob)}")
print(blob.hex(" "))

status=0  len=53
01 00 00 00 00 00 00 00 00 00 00 fe 2a 04 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00


## ST_ON ↔ ST_STBY experiment — KEY QUESTION

1-00 removed the device-level activation state; the main state now reports
`STATE_ON (0)` vs `STATE_STBY (1)` and the hk channel `Activated` derives
from it. **Assumption to verify: `DeviceEnable` of the loaded config drives
the flip.** DLL slot 1 "Standby" has DeviceEnable=N, slot 9 "Heat 30deg"
has DeviceEnable=Y. Also test whether `set_enable()` alone flips it.

⚠️ Prerequisite: main state must NOT be `STATE_ERR_ILOCK` — an interlock
error masks the ON/STBY answer (2026-07-23 run). A working config with
`InterlockEnable=Y,N,N,Y` clears it; if the error persists here, jump to
the working-config section first, then come back.

In [11]:
# CHANGE ESI: load DLL slot 1 "Standby" (file [Configuration2]; DeviceEnable=N, HV modules enabled)
print("load           :", esi.load_current_config(1))
time.sleep(1.0)
print("main state     :", esi.get_main_state())   # expect STATE_STBY?
print("enable         :", esi.get_enable())
print("interlock enab :", esi.get_interlock_enable())
for addr in (2, 3):
    print(f"module {addr} active:", esi.get_module_activation_state(addr))

INFO - esi_100_test  COM14  loading NVM config 1


load           : 0
main state     : (0, '0x1', 'STATE_STBY')
enable         : (0, False)
interlock enab : (0, 6)
module 2 active: (0, True)
module 3 active: (0, True)


In [15]:
# CHANGE ESI: load DLL slot 9 "Heat 30deg" (file [Configuration10]; DeviceEnable=Y, heater 30 degC)
print("load           :", esi.load_current_config(9))
time.sleep(1.0)
print("main state     :", esi.get_main_state())   # expect STATE_ON?
print("enable         :", esi.get_enable())
print("heater target  :", esi.get_heat_ctrl_heater_temperature())
print("power limit    :", esi.get_heat_ctrl_power_limit())  # heat slots should carry 20-30 W
for addr in (2, 3):
    print(f"module {addr} active:", esi.get_module_activation_state(addr))

INFO - esi_100_test  COM14  loading NVM config 9


load           : 0
main state     : (0, '0x0', 'STATE_ON')
enable         : (0, True)
heater target  : (0, 30.0)
power limit    : (0, 19.999515213824)
module 2 active: (0, True)
module 3 active: (0, True)


In [16]:
# CHANGE ESI: set_enable(False) while config 10 is loaded — does enable alone flip ON->STBY?
print("set_enable(F)  :", esi.set_enable(False))
time.sleep(1.0)
print("main state     :", esi.get_main_state())
print("enable         :", esi.get_enable())


INFO - esi_100_test  COM14  setting device enable to False


set_enable(F)  : 0
main state     : (0, '0x1', 'STATE_STBY')
enable         : (0, False)


In [17]:
# CHANGE ESI: set_enable(True) — restore
print("set_enable(T)  :", esi.set_enable(True))
time.sleep(1.0)
print("main state     :", esi.get_main_state())

INFO - esi_100_test  COM14  setting device enable to True


set_enable(T)  : 0
main state     : (0, '0x0', 'STATE_ON')


In [18]:
# Interlock-mask mapping probe: after a config with InterlockEnable=Y,N,N,Y
# is loaded, record the numeric mask (answers cfg-order vs header-bit
# order: header bits are 0=HTCTRL_ILOCK1, 1=HTCTRL_ILOCK2, 2=CTRL_FP, 3=CTRL_RP)
print("interlock enab :", esi.get_interlock_enable())
print("interlock state:", esi.get_interlock_state())
print("main state     :", esi.get_main_state())  # ERR_ILOCK should be gone

interlock enab : (0, 6)
interlock state: (0, '0xfa0e', ['IS_HTCTRL_ILOCK2', 'IS_CTRL_ILOCK_FP', 'IS_CTRL_ILOCK_RP', 'IS_HTCTRL_ILOCK2_CURR', 'IS_HTCTRL_ILOCK2_LAST', 'IS_CTRL_ILOCK_FP_CURR', 'IS_CTRL_ILOCK_RP_CURR', 'IS_CTRL_ILOCK_FP_LAST', 'IS_CTRL_ILOCK_RP_LAST'])
main state     : (0, '0x0', 'STATE_ON')


In [19]:
# CHANGE ESI: back to DLL slot 1 "Standby"
print("load           :", esi.load_current_config(1))
time.sleep(1.0)
print("main state     :", esi.get_main_state())

INFO - esi_100_test  COM14  loading NVM config 1


load           : 0
main state     : (0, '0x1', 'STATE_STBY')


## Working-config workflow (user 2026-07-23) — the everyday use case

Off/Standby → working config → **controller heats + applies HV
immediately** (no extra enable) → tweak temperature and HVPS2/HVPS3
voltages live.

Example: file `[Configuration12]` "Heat 50deg" = **DLL slot 11**
(HeaterTemperature=50, HVPS2Voltage=60, HVPS3Voltage=65,
MaxVoltStep=10, InterlockEnable=Y,N,N,Y, heater limits 22 V / 12 A /
30 W).

If the HV readbacks stay at 0 V here: check the loaded config's
`HVPSxMaxVoltStep` — **0 means no voltage gets applied at all** (10 V is
a good value). That field lives only in the config blob; there is no
dedicated DLL setter.

In [10]:
# CHANGE ESI: load DLL slot 11 "Heat 50deg" (file [Configuration12]) —
# heater 50 degC + HV 60/65 V apply IMMEDIATELY, no further enable
print("name           :", esi.get_config_name(11))
print("load           :", esi.load_current_config(11))
time.sleep(2.0)
print("main state     :", esi.get_main_state())          # expect STATE_ON
print("heater target  :", esi.get_heat_ctrl_heater_temperature())  # expect 50
print("power limit    :", esi.get_heat_ctrl_power_limit())
for addr in (2, 3):
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))
    print(f"HV{addr} V readback :", esi.get_hv_supply_output_voltage(addr))

INFO - esi_100_test  COM14  loading NVM config 11


name           : (0, 'Heat 50deg')
load           : 0
main state     : (0, '0x0', 'STATE_ON')
heater target  : (0, 50.0)
power limit    : (0, 30.000346562559997)
HV2 target     : (0, 60.0)
HV2 V readback : (0, True, 59.273)
HV3 target     : (0, 70.0)
HV3 V readback : (0, True, 61.885)


In [21]:
# CHANGE ESI: tweak on top of the config — heater 50 -> 55 degC,
# HV3 65 -> 70 V (the everyday experiment adjustment)
print("set target     :", esi.set_heat_ctrl_heater_temperature(60.0))
print("set HV3        :", esi.set_hv_supply_target_output_voltage(3, 80.0))
time.sleep(2.0)
print("heater target  :", esi.get_heat_ctrl_heater_temperature())
print("HV3 target     :", esi.get_hv_supply_target_output_voltage(3))
print("HV3 V readback :", esi.get_hv_supply_output_voltage(3))
print("HV2 V readback :", esi.get_hv_supply_output_voltage(2))  # untouched, still ~60 V

INFO - esi_100_test  COM14  setting heater target to 60.0 degC
INFO - esi_100_test  COM14  setting HV module 3 target to 80.00 V


set target     : (0, 60.0)
set HV3        : 0
heater target  : (0, 60.0)
HV3 target     : (0, 80.0)
HV3 V readback : (0, True, 79.908)
HV2 V readback : (0, True, 59.999)


In [22]:
# CHANGE ESI: save the tweaked settings to a scratch NVM slot (PSU/SW-style
# persistence; slot 500 is far away from the user cfg block), name it,
# verify it appears in the catalogue
SCRATCH_SLOT = 500
print("save           :", esi.save_current_config(SCRATCH_SLOT))
print("name           :", esi.set_config_name(SCRATCH_SLOT, "nb032 scratch 55deg"))
print("readback       :", esi.get_config_name(SCRATCH_SLOT))
print("flags          :", esi.get_config_flags(SCRATCH_SLOT))
status, active_slots, _ = esi.list_configs()
print("in catalogue   :", SCRATCH_SLOT in active_slots)

# CHANGE ESI: back to slot 1 "Standby" before the dedicated heater/HV sections
print("load standby   :", esi.load_current_config(1))
time.sleep(1.0)
print("main state     :", esi.get_main_state())

INFO - esi_100_test  COM14  saving current config to NVM slot 500
INFO - esi_100_test  COM14  naming NVM config slot 500 'nb032 scratch 55deg'


save           : 0
name           : 0
readback       : (0, 'nb032 scratch 55deg')
flags          : (0, True, True)


INFO - esi_100_test  COM14  loading NVM config 1


in catalogue   : True
load standby   : 0
main state     : (0, '0x1', 'STATE_STBY')


## Heater (HTCTRL-24-10, address 0)

⚠️ Power limit must stay **10–30 W below 50 °C** (overshoot; 150–180 W is
for ≥50 °C only). Slot 1 "Standby" configures 180 W — the manual-target
cell below sets **20 W** first. Negative target = temperature control off.

In [11]:
print("hw limits      :", esi.get_heat_ctrl_hw_limits())  # (status, maxV, maxA, maxW, maxT)
print("volt limit     :", esi.get_heat_ctrl_voltage_limit())
print("curr limit     :", esi.get_heat_ctrl_current_limit())
print("power limit    :", esi.get_heat_ctrl_power_limit())
print("target temp    :", esi.get_heat_ctrl_heater_temperature())
# (status, valid, volt_out, volt_heat, curr_out, temp_heat)
print("monitoring     :", esi.get_heat_ctrl_monitoring())

hw limits      : (0, 21.999616, 12.000256, 179.99993189171198, 175.0)
volt limit     : (0, 21.999616)
curr limit     : (0, 12.000256)
power limit    : (0, 30.000346562559997)
target temp    : (0, 50.0)
monitoring     : (0, True, 9.045283999999999, 8.84701, 3.4136439999999997, 38.695)


In [12]:
# CHANGE ESI: heater power limit 20 W (mandatory below 50 degC), then target 40 degC
print("set power limit:", esi.set_heat_ctrl_power_limit(20.0))
print("power limit    :", esi.get_heat_ctrl_power_limit())
print("set target     :", esi.set_heat_ctrl_heater_temperature(40.0))
print("target temp    :", esi.get_heat_ctrl_heater_temperature())

INFO - esi_100_test  COM14  setting heater target to 40.0 degC


set power limit: (0, 19.999515213824)
power limit    : (0, 19.999515213824)
set target     : (0, 40.0)
target temp    : (0, 40.0)


In [13]:
# Watch the heater approach the target (read-only; ~2 min at 2 s)
t0 = time.time()
while time.time() - t0 < 120:
    status, valid, vout, vheat, iout, theat = esi.get_heat_ctrl_monitoring()
    power = vout * iout
    print(f"{time.time() - t0:5.0f} s  Temp_Heater={theat:5.1f} degC  "
          f"Vout={vout:5.2f} V  Iout={iout:5.2f} A  P={power:5.1f} W  valid={valid}")
    time.sleep(2.0)

    0 s  Temp_Heater= 39.3 degC  Vout= 0.00 V  Iout= 0.00 A  P=  0.0 W  valid=True
    2 s  Temp_Heater= 39.4 degC  Vout= 0.00 V  Iout= 0.00 A  P=  0.0 W  valid=True
    4 s  Temp_Heater= 39.5 degC  Vout= 0.00 V  Iout= 0.00 A  P=  0.0 W  valid=True
    6 s  Temp_Heater= 39.5 degC  Vout= 0.76 V  Iout= 0.00 A  P=  0.0 W  valid=True
    8 s  Temp_Heater= 39.5 degC  Vout= 2.62 V  Iout= 0.90 A  P=  2.4 W  valid=True
   10 s  Temp_Heater= 39.6 degC  Vout= 2.00 V  Iout= 0.68 A  P=  1.4 W  valid=True
   12 s  Temp_Heater= 39.6 degC  Vout= 2.31 V  Iout= 0.77 A  P=  1.8 W  valid=True
   14 s  Temp_Heater= 39.6 degC  Vout= 2.73 V  Iout= 0.91 A  P=  2.5 W  valid=True
   16 s  Temp_Heater= 39.6 degC  Vout= 3.12 V  Iout= 1.04 A  P=  3.3 W  valid=True
   18 s  Temp_Heater= 39.6 degC  Vout= 3.46 V  Iout= 1.15 A  P=  4.0 W  valid=True
   20 s  Temp_Heater= 39.6 degC  Vout= 3.79 V  Iout= 1.28 A  P=  4.8 W  valid=True
   22 s  Temp_Heater= 39.6 degC  Vout= 4.04 V  Iout= 1.38 A  P=  5.6 W  valid=True
   2

In [ ]:
# CHANGE ESI: heater temperature control OFF (negative target)
print("set target     :", esi.set_heat_ctrl_heater_temperature(-1.0))
print("target temp    :", esi.get_heat_ctrl_heater_temperature())
print("monitoring     :", esi.get_heat_ctrl_monitoring())

## HV modules (2 = inlet, 3 = emitter)

Config first: module enables come from the loaded config; from slot 1
"Standby" both HV modules are enabled but the device sits in STBY. If
targets don't take / outputs stay 0 in STBY, load slot 9 (ON) and retry —
**record which it was**. (Remember `MaxVoltStep=0` in a config also
means 0 V output.)

Meas ranges (CGC email): `NegOut` picks which output (neg/pos) is
measured + regulated (voltage present on BOTH); `HighCurrRange` N ≈ 170 µA
(~35 pArms noise), Y ≈ 1.7 mA (~200 pArms).

In [14]:
esi.load_current_config(11)

INFO - esi_100_test  COM14  loading NVM config 11


0

In [15]:
for addr in (2, 3):
    # (status, volt_neg, curr_high)
    print(f"HV{addr} meas ranges:", esi.get_hv_supply_meas_ranges(addr))
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))
    print(f"HV{addr} V readback :", esi.get_hv_supply_output_voltage(addr))
    print(f"HV{addr} I readback :", esi.get_hv_supply_output_current(addr))

HV2 meas ranges: (0, False, False)
HV2 target     : (0, 60.0)
HV2 V readback : (0, True, 59.999)
HV2 I readback : (0, True, 0.0)
HV3 meas ranges: (0, False, False)
HV3 target     : (0, 70.0)
HV3 V readback : (0, True, 70.0)
HV3 I readback : (0, True, 6.3e-11)


In [34]:
# CHANGE ESI: HV module 2 target +50 V (low first)
print("set target     :", esi.set_hv_supply_target_output_voltage(2, 50.0))
time.sleep(2.0)
print("target readback:", esi.get_hv_supply_target_output_voltage(2))
print("V readback     :", esi.get_hv_supply_output_voltage(2))
print("I readback     :", esi.get_hv_supply_output_current(2))

INFO - esi_100_test  COM14  setting HV module 2 target to 50.00 V


set target     : 0
target readback: (0, 50.0)
V readback     : (0, True, 57.332)
I readback     : (0, True, -1.2519999999999998e-09)


In [35]:
# CHANGE ESI: HV module 2 target +300 V (nb-024 inlet level)
print("set target     :", esi.set_hv_supply_target_output_voltage(2, 300.0))
time.sleep(2.0)
print("target readback:", esi.get_hv_supply_target_output_voltage(2))
print("V readback     :", esi.get_hv_supply_output_voltage(2))
print("I readback     :", esi.get_hv_supply_output_current(2))

INFO - esi_100_test  COM14  setting HV module 2 target to 300.00 V


set target     : 0
target readback: (0, 300.0)
V readback     : (0, True, 58.166000000000004)
I readback     : (0, True, 2.5029999999999997e-09)


In [36]:
# CHANGE ESI: HV module 3 target +50 V, then +300 V
print("set 50 V       :", esi.set_hv_supply_target_output_voltage(3, 50.0))
time.sleep(2.0)
print("V readback     :", esi.get_hv_supply_output_voltage(3))
print("set 300 V      :", esi.set_hv_supply_target_output_voltage(3, 300.0))
time.sleep(2.0)
print("target readback:", esi.get_hv_supply_target_output_voltage(3))
print("V readback     :", esi.get_hv_supply_output_voltage(3))
print("I readback     :", esi.get_hv_supply_output_current(3))

INFO - esi_100_test  COM14  setting HV module 3 target to 50.00 V


set 50 V       : 0


INFO - esi_100_test  COM14  setting HV module 3 target to 300.00 V


V readback     : (0, True, 64.996)
set 300 V      : 0
target readback: (0, 300.0)
V readback     : (0, True, 71.221)
I readback     : (0, True, 2.5029999999999997e-09)


In [37]:
# CHANGE ESI: none — read-only live monitor (60 s at 2 s): HV V/I + heater temp
t0 = time.time()
while time.time() - t0 < 60:
    rows = []
    for addr in (2, 3):
        _, _, volts = esi.get_hv_supply_output_voltage(addr)
        _, _, amps = esi.get_hv_supply_output_current(addr)
        rows.append(f"HV{addr}: {volts:8.2f} V {amps:10.3e} A")
    st, valid, _, _, _, theat = esi.get_heat_ctrl_monitoring()
    rows.append(f"heater: {theat:5.1f} degC" if st == esi.NO_ERR and valid
                else "heater: n/a")
    print("  |  ".join(rows))
    time.sleep(2.0)

HV2:   300.00 V  2.500e-10 A  |  HV3:   299.97 V  1.040e-10 A  |  heater:  50.0 degC
HV2:   300.00 V  2.500e-10 A  |  HV3:   300.00 V  1.250e-10 A  |  heater:  50.0 degC
HV2:   300.00 V  1.880e-10 A  |  HV3:   300.00 V  1.250e-10 A  |  heater:  50.0 degC
HV2:   300.00 V  1.670e-10 A  |  HV3:   300.00 V  1.040e-10 A  |  heater:  50.0 degC
HV2:   300.00 V  2.290e-10 A  |  HV3:   300.00 V  8.300e-11 A  |  heater:  50.0 degC
HV2:   300.00 V  1.670e-10 A  |  HV3:   300.00 V  4.200e-11 A  |  heater:  50.0 degC
HV2:   300.00 V  1.880e-10 A  |  HV3:   300.00 V  1.040e-10 A  |  heater:  50.0 degC
HV2:   300.00 V  1.670e-10 A  |  HV3:   300.00 V  8.300e-11 A  |  heater:  50.0 degC
HV2:   300.00 V  1.880e-10 A  |  HV3:   300.00 V  6.300e-11 A  |  heater:  50.0 degC
HV2:   300.00 V  1.040e-10 A  |  HV3:   300.00 V  1.250e-10 A  |  heater:  50.0 degC
HV2:   300.00 V  2.290e-10 A  |  HV3:   300.00 V  6.300e-11 A  |  heater:  50.0 degC
HV2:   300.00 V  1.670e-10 A  |  HV3:   300.00 V  1.040e-10 A  | 

In [38]:
# CHANGE ESI: module 2 HighCurrRange Y (1.7 mA range), verify, then back to N
print("set ranges     :", esi.set_hv_supply_meas_ranges(2, False, True))
print("readback       :", esi.get_hv_supply_meas_ranges(2))
print("I readback     :", esi.get_hv_supply_output_current(2))  # noise ~200 pArms now?

print("restore ranges :", esi.set_hv_supply_meas_ranges(2, False, False))
print("readback       :", esi.get_hv_supply_meas_ranges(2))

INFO - esi_100_test  COM14  setting HV module 2 meas ranges: volt_neg=False, curr_high=True
INFO - esi_100_test  COM14  setting HV module 2 meas ranges: volt_neg=False, curr_high=False


set ranges     : 0
readback       : (0, False, True)
I readback     : (0, True, -0.002147483648)
restore ranges : 0
readback       : (0, False, False)


In [39]:
# CHANGE ESI: HV targets back to 0 V (both modules)
for addr in (2, 3):
    print(f"HV{addr} -> 0 V    :", esi.set_hv_supply_target_output_voltage(addr, 0.0))
time.sleep(2.0)
for addr in (2, 3):
    print(f"HV{addr} V readback:", esi.get_hv_supply_output_voltage(addr))

INFO - esi_100_test  COM14  setting HV module 2 target to 0.00 V
INFO - esi_100_test  COM14  setting HV module 3 target to 0.00 V


HV2 -> 0 V    : 0
HV3 -> 0 V    : 0
HV2 V readback: (0, True, 250.04500000000002)
HV3 V readback: (0, True, 250.06)


## HV preset mapping check

The presets are named "HV1"/"HV2" and set cfg keys `HVPS1..3` — but the
lab modules sit on addresses **2** and **3** (the HVPS1 slot is
unpopulated, which suggests a direct HVPS-key→address mapping:
HVPS2→2, HVPS3→3). Which address does "HV1 +100V" (DLL slot 100)
actually drive? Read the targets on both modules after loading it.

In [42]:
# CHANGE ESI: load DLL slot 100 "HV1 +100V" (DeviceEnable=Y) — mapping probe
print("name           :", esi.get_config_name(100))
print("load           :", esi.load_current_config(100))
time.sleep(1.0)
print("main state     :", esi.get_main_state())
for addr in (2, 3):
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))
    print(f"HV{addr} V readback :", esi.get_hv_supply_output_voltage(addr))
    print(f"module {addr} active:", esi.get_module_activation_state(addr))

INFO - esi_100_test  COM14  loading NVM config 100


name           : (0, 'HV1 +100V')
load           : 0
main state     : (0, '0x0', 'STATE_ON')
HV2 target     : (0, 0.0)
HV2 V readback : (0, True, 0.006)
module 2 active: (0, False)
HV3 target     : (0, 0.0)
HV3 V readback : (0, True, 0.001)
module 3 active: (0, False)


In [43]:
# CHANGE ESI: back to DLL slot 1 "Standby"
print("load           :", esi.load_current_config(1))
time.sleep(1.0)
print("main state     :", esi.get_main_state())

INFO - esi_100_test  COM14  loading NVM config 1


load           : 0
main state     : (0, '0x1', 'STATE_STBY')


## HV targets in STBY — do they take effect? (added 2026-07-27)

Open question from the first run. Trick: reach STBY via `set_enable(False)`
on top of slot 11 "Heat 50deg" (MaxVoltStep=10) — the Standby config itself
has MaxVoltStep=0, which would mask any output regardless of the answer.
Distinct targets (55/70 V, not the config's 60/65) make a later overwrite
by a config load visible. Answers wanted: **(a)** do set-target calls
return status 0 in STBY, **(b)** is output produced in STBY, **(c)** do
stored targets drive output after `set_enable(True)`, **(d)** does a
config load overwrite live targets with its presets?

In [10]:
esi.load_current_config(11)

INFO - esi_100_test  COM14  loading NVM config 11


0

In [11]:
# CHANGE ESI: load DLL slot 11 "Heat 50deg" (MaxVoltStep=10), then set_enable(False) -> STBY
print("load           :", esi.load_current_config(11))
time.sleep(1.0)
print("set_enable(F)  :", esi.set_enable(False))
time.sleep(1.0)
print("main state     :", esi.get_main_state())   # expect ST_STBY
for addr in (2, 3):
    print(f"module {addr} active:", esi.get_module_activation_state(addr))

INFO - esi_100_test  COM14  loading NVM config 11


load           : 0


INFO - esi_100_test  COM14  setting device enable to False


set_enable(F)  : 0
main state     : (0, '0x1', 'STATE_STBY')
module 2 active: (0, True)
module 3 active: (0, True)


In [12]:
# CHANGE ESI: set distinct HV targets IN STBY — 55 V (mod 2) / 70 V (mod 3)
print("set HV2 55 V   :", esi.set_hv_supply_target_output_voltage(2, 55.0))
print("set HV3 70 V   :", esi.set_hv_supply_target_output_voltage(3, 70.0))
time.sleep(2.0)
for addr in (2, 3):
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))
    print(f"HV{addr} V readback :", esi.get_hv_supply_output_voltage(addr))
# (a) status 0? (b) target stored? V readback in STBY presumably 0 — confirm

INFO - esi_100_test  COM14  setting HV module 2 target to 55.00 V
INFO - esi_100_test  COM14  setting HV module 3 target to 70.00 V


set HV2 55 V   : 0
set HV3 70 V   : 0
HV2 target     : (0, 55.0)
HV2 V readback : (0, True, 10.363)
HV3 target     : (0, 70.0)
HV3 V readback : (0, True, 11.561)


In [13]:
# CHANGE ESI: set_enable(True) -> ST_ON — do the stored targets now drive output?
print("set_enable(T)  :", esi.set_enable(True))
time.sleep(2.0)
print("main state     :", esi.get_main_state())
for addr in (2, 3):
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))
    print(f"HV{addr} V readback :", esi.get_hv_supply_output_voltage(addr))   # 55/70 V or config 60/65?

INFO - esi_100_test  COM14  setting device enable to True


set_enable(T)  : 0
main state     : (0, '0x0', 'STATE_ON')
HV2 target     : (0, 55.0)
HV2 V readback : (0, True, 55.845)
HV3 target     : (0, 70.0)
HV3 V readback : (0, True, 65.461)


In [14]:
# CHANGE ESI: reload slot 11 — does a config load overwrite live targets with its presets?
print("load           :", esi.load_current_config(11))
time.sleep(2.0)
for addr in (2, 3):
    print(f"HV{addr} target     :", esi.get_hv_supply_target_output_voltage(addr))   # back to 60/65?

INFO - esi_100_test  COM14  loading NVM config 11


load           : 0
HV2 target     : (0, 60.0)
HV3 target     : (0, 70.0)


In [15]:
# CHANGE ESI: targets 0 V, back to DLL slot 1 "Standby" — park for the Fan section
for addr in (2, 3):
    print(f"HV{addr} -> 0 V    :", esi.set_hv_supply_target_output_voltage(addr, 0.0))
print("load           :", esi.load_current_config(1))
time.sleep(1.0)
print("main state     :", esi.get_main_state())

INFO - esi_100_test  COM14  setting HV module 2 target to 0.00 V
INFO - esi_100_test  COM14  setting HV module 3 target to 0.00 V
INFO - esi_100_test  COM14  loading NVM config 1


HV2 -> 0 V    : 0
HV3 -> 0 V    : 0
load           : 0
main state     : (0, '0x1', 'STATE_STBY')


## Fan slide-switch override (new in 1-00)

Safe per CGC email: fan is temperature-driven (runs from ≥30 °C) and the
device deactivates on critical temperature even with the fan forced off.

In [44]:
print("fan state      :", esi.get_fan_state())
print("fan data       :", esi.get_fan_data())  # (status, failed, max, set, meas, pwm)
print("fan override   :", esi.get_fan_switch_override())

fan state      : (0, '0xf', ['FS_FAN_OK', 'FS_FAN_SW_CURR', 'FS_FAN_SW_LAST', 'FS_FAN_ENB'])
fan data       : (0, False, 6000, 0, 0, 0.0)
fan override   : (0, 0)


In [51]:
# CHANGE ESI: force fan ON via override, verify, then release override
print("override ON    :", esi.set_fan_switch_override(esi.FS_OVERRIDE | esi.FS_ON))
time.sleep(2.0)
print("fan override   :", esi.get_fan_switch_override())
print("fan data       :", esi.get_fan_data())

print("release        :", esi.set_fan_switch_override(0))
print("fan override   :", esi.get_fan_switch_override())

override ON    : 0
fan override   : (0, 3)
fan data       : (0, False, 6000, 0, 0, 0.0)
release        : 0
fan override   : (0, 0)


## Housekeeping cycle

One manual `hk_monitor()` cycle — canonical log lines for every channel,
including the 1-00 `Activated` derivation (`main_state == STATE_ON`) and
the new `Temp_Heater`. With slot 1 "Standby" loaded, expect `Activated 0`.

In [47]:
esi.hk_monitor()

INFO - esi_100_test  COM14  Volt_24V             24.11 V
INFO - esi_100_test  COM14  Volt_5V0             5.01 V
INFO - esi_100_test  COM14  Volt_3V3             3.30 V
INFO - esi_100_test  COM14  Temp_CPU             26.2 degC
INFO - esi_100_test  COM14  Temp_PSU             27.7 degC
INFO - esi_100_test  COM14  CPU_Load             0.0 %
INFO - esi_100_test  COM14  Fan_RPM              0 rpm
INFO - esi_100_test  COM14  Main_State           STATE_STBY
INFO - esi_100_test  COM14  Activated            0
INFO - esi_100_test  COM14  Device_State         DEVST_OK
INFO - esi_100_test  COM14  Enabled              0
INFO - esi_100_test  COM14  Temp_Heater          42.7 degC
INFO - esi_100_test  COM14  Modules_Present      2
INFO - esi_100_test  COM14  HV2_Voltage          -0.04 V
INFO - esi_100_test  COM14  HV2_Current          1.250e-10 A
INFO - esi_100_test  COM14  HV3_Voltage          -0.04 V
INFO - esi_100_test  COM14  HV3_Current          -4.200e-11 A


## Safe shutdown

Park the device: HV targets are already 0 V; load DLL slot 0 "Off"
(everything disabled), then disconnect.

In [16]:
# CHANGE ESI: load DLL slot 0 "Off" (device + modules + heater all disabled)
print("load           :", esi.load_current_config(0))
time.sleep(1.0)
print("main state     :", esi.get_main_state())
print("enable         :", esi.get_enable())

INFO - esi_100_test  COM14  loading NVM config 0


load           : 0
main state     : (0, '0x1', 'STATE_STBY')
enable         : (0, False)


In [17]:
esi.disconnect()

INFO - esi_100_test  COM14  disconnected


True

## Findings (fill in on hardware)

- **32-bit DLL bridge:** identity/state/config reads OK over the bridge?
  latency acceptable? _______
- **Interlock:** working config cleared `STATE_ERR_ILOCK`? numeric mask
  for `Y,N,N,Y` = ___ (header bits: 0=HTCTRL_ILOCK1, 1=HTCTRL_ILOCK2,
  2=CTRL_FP, 3=CTRL_RP)
- **ST_ON ↔ ST_STBY driver:** config `DeviceEnable`? `set_enable()`?
  → _______
- **`set_enable(False)` while ON:** state became _______ (does bring-up's
  `set_enable(True)` interfere with config semantics?)
- **Working config (slot 11 "Heat 50deg"):** heating + HV applied
  immediately on load with no extra enable? HV2/HV3 read ~60/65 V? ___
  Tweaks (55 °C, HV3 70 V) took while ON? ___
- **Save/name config:** scratch slot 500 saved + named + listed? ___
- **Preset mapping:** slot 100 "HV1 +100V" drove address ___ (module
  enables after load: 2 = ___, 3 = ___)
- **Do HV targets take in STBY** (slot 1), or only ON? → _______
- **Slot numbering:** file `[ConfigurationN]` = DLL slot N−1 confirmed? ___
- **Signature fixes:** base-hk Valid = ___, complete-state values sane? ___
- **comspeed:** bring-up log line said actual baud = _______
- **Heater:** 40 °C reached in ___ s at 20 W; overshoot ___ °C;
  `Temp_Heater` in hk = OK?
- **Meas-range toggle:** current-noise change visible? _______
- Anything unexpected → [[cgc-esi]] quirks section.
